<a href="https://colab.research.google.com/github/abdullah2709/EAAI_Project_2/blob/main/project_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Section 13: Hugging Face Style Project Dashboard ---

from IPython.display import display, HTML
import pandas as pd
import numpy as np

# -----------------------------
# 1. Safely collect project facts
# -----------------------------
n_rows = len(df) if 'df' in globals() else None
n_features = X_train.shape[1] if 'X_train' in globals() else None
train_rows = len(X_train) if 'X_train' in globals() else None
test_rows = len(X_test) if 'X_test' in globals() else None

best_overall = results_df["RMSE"].idxmin() if 'results_df' in globals() else "Not available"
best_rmse = results_df["RMSE"].min() if 'results_df' in globals() else None
best_mae = results_df.loc[best_overall, "MAE"] if 'results_df' in globals() else None
best_r2 = results_df.loc[best_overall, "R2"] if 'results_df' in globals() else None
best_mape = results_df.loc[best_overall, "MAPE"] if 'results_df' in globals() and "MAPE" in results_df.columns else None

peak_model = results_df["Peak_MAE"].idxmin() if 'results_df' in globals() and "Peak_MAE" in results_df.columns else "Not available"
peak_mae = results_df["Peak_MAE"].min() if 'results_df' in globals() and "Peak_MAE" in results_df.columns else None

wf_mean_rmse = wf_df["RMSE"].mean() if 'wf_df' in globals() and "RMSE" in wf_df.columns else None
wf_cv = (wf_df["RMSE"].std() / wf_df["RMSE"].mean() * 100) if 'wf_df' in globals() and "RMSE" in wf_df.columns else None

target_name = "load_MW"
dataset_type = "Multivariate Supervised Time-Series Forecasting"
project_title = "AI-Based Energy Consumption Forecasting"
primary_use = "Hourly electricity demand prediction with weather-enhanced forecasting"

# -----------------------------
# 2. Build feature summary
# -----------------------------
feature_groups = {
    "Calendar Features": [],
    "Cyclical Features": [],
    "Lag Features": [],
    "Rolling Features": [],
    "Weather Features": [],
    "Other Features": []
}

if 'X_train' in globals():
    for col in X_train.columns:
        col_lower = col.lower()
        if any(k in col_lower for k in ["hour", "dayofweek", "weekday", "month", "weekend", "holiday"]):
            feature_groups["Calendar Features"].append(col)
        elif any(k in col_lower for k in ["sin", "cos"]):
            feature_groups["Cyclical Features"].append(col)
        elif "lag_" in col_lower:
            feature_groups["Lag Features"].append(col)
        elif any(k in col_lower for k in ["rolling", "roll_", "mean_", "std_"]):
            feature_groups["Rolling Features"].append(col)
        elif any(k in col_lower for k in ["temp", "humidity", "wind", "pressure", "precip"]):
            feature_groups["Weather Features"].append(col)
        else:
            feature_groups["Other Features"].append(col)

def format_feature_group(items, max_show=8):
    if len(items) == 0:
        return "None"
    if len(items) <= max_show:
        return ", ".join(items)
    return ", ".join(items[:max_show]) + f", ... (+{len(items)-max_show} more)"

# -----------------------------
# 3. Prepare model leaderboard HTML
# -----------------------------
leaderboard_html = ""
if 'results_df' in globals():
    leaderboard_df = results_df.copy()

    preferred_cols = [c for c in ["MAE", "RMSE", "R2", "MAPE", "Peak_MAE", "Improvement_%"] if c in leaderboard_df.columns]
    leaderboard_df = leaderboard_df[preferred_cols].copy()

    # Round values for cleaner display
    for col in leaderboard_df.columns:
        if leaderboard_df[col].dtype != "O":
            leaderboard_df[col] = leaderboard_df[col].round(3)

    rows_html = ""
    for model_name, row in leaderboard_df.iterrows():
        badge = "🏆 Best RMSE" if model_name == best_overall else ""
        peak_badge = "⚡ Peak Winner" if model_name == peak_model else ""
        extra_badge = f"{badge} {peak_badge}".strip()

        row_cells = "".join([f"<td>{row[c]}</td>" for c in leaderboard_df.columns])
        rows_html += f"""
        <tr>
            <td><b>{model_name}</b></td>
            {row_cells}
            <td>{extra_badge}</td>
        </tr>
        """

    header_cells = "".join([f"<th>{c}</th>" for c in leaderboard_df.columns])

    leaderboard_html = f"""
    <table class="hf-table">
        <thead>
            <tr>
                <th>Model</th>
                {header_cells}
                <th>Status</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
    """
else:
    leaderboard_html = "<p>Leaderboard not available.</p>"

# -----------------------------
# 4. Final recommendation text
# -----------------------------
if best_overall == peak_model and best_overall != "Not available":
    recommendation = f"""
    <b>Recommended model:</b> {best_overall}<br>
    This model achieved the strongest overall forecasting accuracy and also performed best under peak-demand conditions,
    making it the most balanced option for deployment.
    """
elif best_overall != "Not available" and peak_model != "Not available":
    recommendation = f"""
    <b>Recommended deployment model:</b> {peak_model}<br>
    Although <b>{best_overall}</b> achieved the best average RMSE, peak-demand resilience is more critical in an energy forecasting context.
    Therefore, <b>{peak_model}</b> is preferred for operational use.
    """
else:
    recommendation = """
    Final recommendation could not be generated automatically because one or more evaluation objects are missing.
    """

# -----------------------------
# 5. Render Hugging Face style dashboard
# -----------------------------
html = f"""
<style>
    .hf-wrapper {{
        font-family: Arial, sans-serif;
        max-width: 1200px;
        margin: 20px auto;
        padding: 24px;
        background: #0f172a;
        color: #e5e7eb;
        border-radius: 18px;
        box-shadow: 0 8px 24px rgba(0,0,0,0.25);
    }}

    .hf-title {{
        font-size: 32px;
        font-weight: 800;
        margin-bottom: 6px;
    }}

    .hf-subtitle {{
        font-size: 15px;
        color: #cbd5e1;
        margin-bottom: 22px;
    }}

    .hf-pill {{
        display: inline-block;
        background: #1e293b;
        color: #93c5fd;
        padding: 6px 12px;
        border-radius: 999px;
        margin-right: 8px;
        margin-bottom: 8px;
        font-size: 13px;
        font-weight: 600;
    }}

    .hf-grid {{
        display: grid;
        grid-template-columns: repeat(4, 1fr);
        gap: 14px;
        margin-top: 18px;
        margin-bottom: 24px;
    }}

    .hf-card {{
        background: #111827;
        border: 1px solid #334155;
        border-radius: 14px;
        padding: 16px;
    }}

    .hf-card h3 {{
        margin: 0 0 10px 0;
        font-size: 14px;
        color: #93c5fd;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }}

    .hf-big {{
        font-size: 24px;
        font-weight: 800;
        color: #f8fafc;
    }}

    .hf-small {{
        font-size: 13px;
        color: #cbd5e1;
        margin-top: 6px;
    }}

    .hf-section {{
        margin-top: 24px;
    }}

    .hf-section h2 {{
        font-size: 20px;
        margin-bottom: 12px;
        border-bottom: 1px solid #334155;
        padding-bottom: 8px;
    }}

    .hf-table {{
        width: 100%;
        border-collapse: collapse;
        margin-top: 10px;
        background: #111827;
        border-radius: 12px;
        overflow: hidden;
    }}

    .hf-table th, .hf-table td {{
        border: 1px solid #334155;
        padding: 10px;
        text-align: center;
        font-size: 14px;
    }}

    .hf-table th {{
        background: #1e293b;
        color: #93c5fd;
    }}

    .hf-two-col {{
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 16px;
    }}

    .hf-list {{
        margin: 0;
        padding-left: 20px;
    }}

    .hf-list li {{
        margin-bottom: 8px;
    }}

    .hf-footer {{
        margin-top: 24px;
        padding: 16px;
        background: #1e293b;
        border-left: 5px solid #22c55e;
        border-radius: 12px;
        color: #f8fafc;
        line-height: 1.6;
    }}
</style>

<div class="hf-wrapper">

    <div class="hf-title">🤗 {project_title}</div>
    <div class="hf-subtitle">
        Hugging Face style project dashboard for the final notebook
    </div>

    <div>
        <span class="hf-pill">Task: Regression</span>
        <span class="hf-pill">Dataset Type: {dataset_type}</span>
        <span class="hf-pill">Target: {target_name}</span>
        <span class="hf-pill">Use Case: Energy Forecasting</span>
    </div>

    <div class="hf-grid">
        <div class="hf-card">
            <h3>Best Model</h3>
            <div class="hf-big">{best_overall}</div>
            <div class="hf-small">Selected using lowest RMSE</div>
        </div>

        <div class="hf-card">
            <h3>Best RMSE</h3>
            <div class="hf-big">{round(best_rmse, 2) if best_rmse is not None else "N/A"}</div>
            <div class="hf-small">Lower is better</div>
        </div>

        <div class="hf-card">
            <h3>Best R²</h3>
            <div class="hf-big">{round(best_r2, 4) if best_r2 is not None else "N/A"}</div>
            <div class="hf-small">Explained variance</div>
        </div>

        <div class="hf-card">
            <h3>Peak Winner</h3>
            <div class="hf-big">{peak_model}</div>
            <div class="hf-small">Most resilient under demand stress</div>
        </div>
    </div>

    <div class="hf-grid">
        <div class="hf-card">
            <h3>Total Samples</h3>
            <div class="hf-big">{n_rows if n_rows is not None else "N/A"}</div>
            <div class="hf-small">Rows in merged dataset</div>
        </div>

        <div class="hf-card">
            <h3>Feature Count</h3>
            <div class="hf-big">{n_features if n_features is not None else "N/A"}</div>
            <div class="hf-small">Model input variables</div>
        </div>

        <div class="hf-card">
            <h3>Walk-Forward RMSE</h3>
            <div class="hf-big">{round(wf_mean_rmse, 2) if wf_mean_rmse is not None else "N/A"}</div>
            <div class="hf-small">Average temporal validation error</div>
        </div>

        <div class="hf-card">
            <h3>Stability (CV %)</h3>
            <div class="hf-big">{round(wf_cv, 2) if wf_cv is not None else "N/A"}</div>
            <div class="hf-small">Lower indicates more stable forecasting</div>
        </div>
    </div>

    <div class="hf-section">
        <h2>Model Summary</h2>
        <div class="hf-card">
            This project predicts <b>hourly electricity demand</b> using historical energy load and weather variables.
            The final analytical problem is a <b>multivariate supervised time-series regression task</b>, where the
            target variable is continuous and forecasting accuracy is assessed using regression metrics.
        </div>
    </div>

    <div class="hf-section">
        <h2>Model Leaderboard</h2>
        {leaderboard_html}
    </div>

    <div class="hf-section hf-two-col">
        <div class="hf-card">
            <h3>Dataset Overview</h3>
            <ul class="hf-list">
                <li><b>Energy dataset:</b> Univariate time-series (hourly demand)</li>
                <li><b>Weather dataset:</b> Multivariate time-series (temperature, humidity, wind, pressure, precipitation)</li>
                <li><b>Merged dataset:</b> Multivariate supervised forecasting dataset</li>
                <li><b>Primary use:</b> {primary_use}</li>
                <li><b>Training rows:</b> {train_rows if train_rows is not None else "N/A"}</li>
                <li><b>Test rows:</b> {test_rows if test_rows is not None else "N/A"}</li>
            </ul>
        </div>

        <div class="hf-card">
            <h3>Feature Engineering Summary</h3>
            <ul class="hf-list">
                <li><b>Calendar Features:</b> {format_feature_group(feature_groups["Calendar Features"])}</li>
                <li><b>Cyclical Features:</b> {format_feature_group(feature_groups["Cyclical Features"])}</li>
                <li><b>Lag Features:</b> {format_feature_group(feature_groups["Lag Features"])}</li>
                <li><b>Rolling Features:</b> {format_feature_group(feature_groups["Rolling Features"])}</li>
                <li><b>Weather Features:</b> {format_feature_group(feature_groups["Weather Features"])}</li>
                <li><b>Other Features:</b> {format_feature_group(feature_groups["Other Features"])}</li>
            </ul>
        </div>
    </div>

    <div class="hf-section hf-two-col">
        <div class="hf-card">
            <h3>Strengths</h3>
            <ul class="hf-list">
                <li>Combines historical load behaviour with external weather signals</li>
                <li>Uses time-aware feature engineering appropriate for forecasting</li>
                <li>Benchmarks multiple regression models rather than relying on one method</li>
                <li>Includes residual analysis, peak-demand testing, and walk-forward validation</li>
            </ul>
        </div>

        <div class="hf-card">
            <h3>Limitations</h3>
            <ul class="hf-list">
                <li>Performance may degrade during rare or abnormal events</li>
                <li>Results depend on correct time alignment across energy and weather data</li>
                <li>Findings are most reliable for the region and period represented in the dataset</li>
                <li>Operational deployment still requires monitoring and periodic retraining</li>
            </ul>
        </div>
    </div>

    <div class="hf-section">
        <h2>Why Regression Instead of Classification or Clustering?</h2>
        <div class="hf-card">
            The project target, <b>{target_name}</b>, is a continuous numerical variable rather than a category.
            For that reason, the task is naturally a <b>regression problem</b>. Classification would require
            artificially converting demand into labels such as "low", "medium", or "high", which would discard
            useful information. Clustering may help with exploratory analysis, but it does not directly solve the
            forecasting objective.
        </div>
    </div>

    <div class="hf-footer">
        {recommendation}
    </div>

</div>
"""

display(HTML(html))